# rai-microrts-arena Kaggle T4x2 training script

This notebook uses Kaggle's active Python/CUDA/Torch runtime directly. The repo metadata may still say `python <3.12`, so on Python >= 3.12 the notebook uses `PYTHONPATH` instead of creating a separate environment.

Current strongest research candidate: `Microrts-kaggle-hierarchical-hybrid-memory-map16-bots`. It combines SquNet tactical grid features, entity graph context, region tokens, and GRU strategic memory while keeping the primitive GridNet action head and legal action masks. `Microrts-kaggle-squnet-map16-bots` remains the proven baseline; `Microrts-kaggle-hybrid-map16-bots` is the lighter entity-grid ablation.

Run flow:

1. Check GPU, Java, Python, and CUDA.
2. Use Kaggle's current kernel; do not install uv or create another Python environment.
3. Install SB3 and microRTS dependencies; native packages try wheels first and local builds if needed.
4. Clone/update the repo and append Kaggle-friendly PPO configs.
5. Profile `squeeze_unet`, `hybrid_entity_grid`, and `hierarchical_hybrid_entity_grid` before training.
6. Run the strongest candidate on one GPU by default; enable dual T4 runs for baseline comparison.

> This is not PPO DDP. On T4x2 the stable pattern is two independent runs: GPU0 for SquNet baseline and GPU1 for HierarchicalHybridMemory.


## Metrics to watch

**Evaluation / win rate**: watch `eval/score`, `WinLoss`, and bot-specific results for Mayari, Coac, WorkerRush, and LightRush. Shaped reward alone can make a weak policy look healthy.

**PPO stability**: watch `approx_kl`, `clip_fraction`, `entropy`, `explained_variance`, `policy_loss`, `value_loss`, and `grad_norm`. KL near zero for a long time usually means updates are too weak; entropy collapsing early usually means exploration is too low.

**Action mask / environment health**: watch `action_mask_stats/valid_locs`, `action_mask_stats/no_valid`, episode length, and timeout/truncation. `no_valid` should stay near zero.

**Memory interface health**: for the hierarchical run, watch for abnormal KL, large eval/rollout mismatch, or long-game failures after episode resets. If it is unstable, fall back to `Microrts-kaggle-hybrid-map16-bots` and compare.

**Throughput bottleneck**: watch `steps_per_second`, GPU util, GPU memory, and `scripts/profile_microrts_policy.py` `p95_ms`. Kaggle microRTS is often CPU/Java-env limited rather than GPU limited.

**Tuning order**: adjust `n_envs`, `n_steps`, `batch_size`, `n_epochs`, `learning_rate`, `clip_range`, `ent_coef`, `vf_coef`, `gamma`, then `gae_lambda`. Stabilize KL and win rate before chasing throughput.


In [ ]:
# 0. Runtime sanity check. This is Kaggle's system Python.
import os, sys, subprocess, textwrap, json
from pathlib import Path

print('system python:', sys.version)
print('Kaggle working dir:', Path('/kaggle/working').exists())
print('Kaggle input dir:', Path('/kaggle/input').exists())
!nvidia-smi
!java -version || true


In [ ]:
# 1. User config
from pathlib import Path
import os, sys

REPO_URL = 'https://github.com/SShion0721/rai-microrts-arena.git'
BRANCH = 'main'
WORKDIR = Path('/kaggle/working/rai-microrts-arena')
PYTHON_SENTINEL = Path('/kaggle/working/rai_microrts_python.txt')

# Always use Kaggle's active notebook Python/CUDA/Torch environment.
KAGGLE_PY = Path(sys.executable)

# Start small. Increase to 10e6/20e6 only after the smoke run is healthy.
KAGGLE_TIMESTEPS = '2e6'
# Set True after the first healthy smoke run; first compile can be quiet/slow.
KAGGLE_TORCH_COMPILE = False
WANDB_PROJECT = 'rai-microrts-kaggle'
USE_WANDB = False

BASELINE_EXPERIMENT = 'Microrts-kaggle-squnet-map16-bots'
ABLATION_EXPERIMENT = 'Microrts-kaggle-hybrid-map16-bots'
STRONGEST_EXPERIMENT = 'Microrts-kaggle-hierarchical-hybrid-memory-map16-bots'
PRIMARY_EXPERIMENT = STRONGEST_EXPERIMENT

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['WORKDIR'] = str(WORKDIR)
os.environ['PYTHONPATH'] = str(WORKDIR) + ':' + os.environ.get('PYTHONPATH', '')
os.environ['KAGGLE_TIMESTEPS'] = KAGGLE_TIMESTEPS
os.environ['KAGGLE_TORCH_COMPILE'] = str(KAGGLE_TORCH_COMPILE).lower()
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
os.environ['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
os.environ['PYTHONUNBUFFERED'] = '1'
print('WORKDIR =', WORKDIR)
print('KAGGLE_PY =', KAGGLE_PY)
print('PRIMARY_EXPERIMENT =', PRIMARY_EXPERIMENT)
print('PYTHONPATH starts with WORKDIR =', os.environ['PYTHONPATH'].startswith(str(WORKDIR)))
print('WANDB_MODE =', os.environ['WANDB_MODE'])
print('KAGGLE_TORCH_COMPILE =', os.environ['KAGGLE_TORCH_COMPILE'])
print('PYTHONUNBUFFERED =', os.environ['PYTHONUNBUFFERED'])


In [ ]:
%%bash
set -Eeuo pipefail
trap 'rc=$?; echo; echo "FAILED at line $LINENO"; echo "Command: $BASH_COMMAND"; echo "Exit code: $rc"; exit $rc' ERR

# Use Kaggle's existing Python/CUDA/Torch environment. Do not install uv or create another Python environment.
# Clear stale variables left by earlier notebook runs.
unset PYTHON_BIN USE_KAGGLE_ENV || true
export REPO_URL="${REPO_URL:-https://github.com/SShion0721/rai-microrts-arena.git}"
export BRANCH="${BRANCH:-main}"
export WORKDIR="${WORKDIR:-/kaggle/working/rai-microrts-arena}"
export KAGGLE_PY="$(command -v python)"
export PYTHONPATH="$WORKDIR:${PYTHONPATH:-}"
export PATH="$HOME/.local/bin:$PATH"
export MAKEFLAGS="-j$(nproc)"
export CMAKE_BUILD_PARALLEL_LEVEL="$(nproc)"
export PYTHON_SENTINEL="/kaggle/working/rai_microrts_python.txt"

echo "$KAGGLE_PY" > "$PYTHON_SENTINEL"
echo "Selected Kaggle Python: $KAGGLE_PY"
"$KAGGLE_PY" -V
"$KAGGLE_PY" -m pip -V
nvidia-smi || true

# Build/runtime system packages. Kaggle often already has many of these.
if command -v apt-get >/dev/null 2>&1; then
  apt-get update -qq || true
  apt-get install -y -qq \
    default-jdk xvfb ffmpeg git unzip \
    build-essential python3-dev cmake ninja-build swig pkg-config || true
fi
command -v java
java -version

if [ ! -d "$WORKDIR/.git" ]; then
  git clone --branch "$BRANCH" --depth 1 "$REPO_URL" "$WORKDIR"
else
  cd "$WORKDIR"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull --ff-only || true
fi

cd "$WORKDIR"

# Upgrade only the build frontend. Keep Kaggle's torch/torchvision/CUDA stack as-is.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python -U pip setuptools wheel build cython packaging ninja

# Python >=3.12 cannot satisfy this repo's current pyproject Requires-Python (<3.12).
# Use PYTHONPATH fallback instead of failing editable install.
if "$KAGGLE_PY" - <<'PYCHECK'
import sys
raise SystemExit(0 if sys.version_info < (3, 12) else 1)
PYCHECK
then
  "$KAGGLE_PY" -m pip install --no-cache-dir -e . --no-deps || echo "editable install failed; using PYTHONPATH fallback"
else
  echo "Python >=3.12 detected; skipping editable install and using PYTHONPATH fallback."
fi

# Minimal runtime deps. Do not install torch/torchvision here; use Kaggle's preinstalled CUDA build.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --prefer-binary \
  'numpy<2' \
  'gymnasium==0.29.1' \
  'stable-baselines3==2.1.0' \
  'ray[air]>=2.8.1,<2.40' \
  'moviepy<2' \
  wandb tensorboard accelerate einops GPUtil \
  pyvirtualdisplay PyYAML tqdm psutil pandas matplotlib

# Native / sometimes-problematic deps. Try wheels first; if unavailable, build locally inside Kaggle.
"$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --prefer-binary \
  'JPype1>=1.3,<2' 'peewee>=3.14.8' 'PettingZoo==1.24.3' || {
    echo 'Binary install failed; trying local source builds for native deps.'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --no-binary=JPype1 'JPype1>=1.3,<2'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python --no-binary=peewee 'peewee>=3.14.8'
    "$KAGGLE_PY" -m pip install --no-cache-dir --ignore-requires-python 'PettingZoo==1.24.3'
  }

PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" "$KAGGLE_PY" - <<'PYCHK'
import sys
import torch
import gymnasium
import stable_baselines3
import yaml
import jpype
import ray
import rl_algo_impls
print('python executable:', sys.executable)
print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
print('gymnasium:', gymnasium.__version__)
print('stable_baselines3:', stable_baselines3.__version__)
PYCHK


In [ ]:
# 2. Import check in the selected Kaggle Python
from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()
print('Using Kaggle Python:', KAGGLE_PY)
!cd "$WORKDIR" && PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" $KAGGLE_PY - <<'PY'
import sys
import torch
import gymnasium
import stable_baselines3
import yaml
import jpype
import ray
import rl_algo_impls
print('python executable:', sys.executable)
print('python version:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
print('gymnasium:', gymnasium.__version__)
print('stable_baselines3:', stable_baselines3.__version__)
PY


In [ ]:
# 3. Append Kaggle-friendly PPO configs. This only changes the notebook runtime copy.
from pathlib import Path

hp_path = WORKDIR / 'rl_algo_impls' / 'hyperparams' / 'ppo-Microrts.yml'
text = hp_path.read_text(encoding='utf-8')

KAGGLE_CONFIG = f"""
# ---- Kaggle T4x2 smoke/ablation configs appended by kaggle_microrts_t4x2.ipynb ----
Microrts-kaggle-squnet-map16-bots: &microrts-kaggle-squnet-map16-bots
  <<: *microrts-squnet-map16
  n_timesteps: !!float {KAGGLE_TIMESTEPS}
  evaluate_after_training: true
  # Keep Kaggle smoke runs static; inherited transitions may omit multi_reward_weights.
  hyperparam_transitions_kwargs: {{}}
  device_hyperparams:
    set_float32_matmul_precision: high
    use_deterministic_algorithms: false
    # Keep smoke runs transparent; set KAGGLE_TORCH_COMPILE=True after the first healthy run.
    torch_compile: {str(KAGGLE_TORCH_COMPILE).lower()}
    compile_mode: reduce-overhead
    cudnn_benchmark: true
    cudnn_allow_tf32: true
    cuda_matmul_allow_tf32: true
  env_hyperparams:
    <<: *microrts-squnet-map16-env-defaults
    n_envs: 12
    self_play_kwargs: null
    map_paths:
      - maps/16x16/basesWorkers16x16A.xml
      - maps/16x16/TwoBasesBarracks16x16.xml
      - maps/16x16/melee16x16Mixed12.xml
    make_kwargs:
      <<: *microrts-squnet-map16-env-make-kwargs-defaults
      num_selfplay_envs: 0
      num_bot_envs: 12
      max_steps: 3000
    bots:
      coacAI: 6
      mayari: 6
  rollout_hyperparams:
    <<: *microrts-ai-rollout-defaults
    n_steps: 512
  algo_hyperparams:
    <<: *microrts-squnet-map16-algo-defaults
    batch_size: 3072
    n_epochs: 4
    learning_rate: !!float 1e-4
    clip_range: 0.1
    ent_coef: 0.01
    # microRTS returns shaped, win/loss, and score-delta rewards. PPO policy loss needs a single advantage.
    multi_reward_weights: [0.8, 0.01, 0.19]
    vf_coef: [0.5, 0.1, 0.2]
  eval_hyperparams:
    <<: *microrts-squnet-map16-eval-defaults
    step_freq: !!float 2.5e5
    n_episodes: 12
    disable_video_generation: true
    skip_evaluate_at_start: true
    env_overrides:
      <<: *microrts-squnet-map16-eval-env-overrides
      n_envs: 12
      self_play_kwargs: {{}}
      bots:
        coacAI: 3
        mayari: 3
        workerRushAI: 3
        lightRushAI: 3

Microrts-kaggle-hybrid-map16-bots:
  <<: *microrts-kaggle-squnet-map16-bots
  policy_hyperparams:
    <<: *microrts-squnet-map16-policy-defaults
    actor_head_style: hybrid_entity_grid
    normalization: layer
    encoder_embed_dim: 128
    encoder_attention_heads: 4
    encoder_feed_forward_dim: 256
    encoder_layers: 2
    actor_head_kernel_size: 3

Microrts-kaggle-hierarchical-hybrid-memory-map16-bots:
  <<: *microrts-kaggle-squnet-map16-bots
  policy_hyperparams:
    <<: *microrts-squnet-map16-policy-defaults
    actor_head_style: hierarchical_hybrid_entity_grid
    normalization: layer
    encoder_embed_dim: 128
    encoder_attention_heads: 4
    encoder_feed_forward_dim: 256
    encoder_layers: 2
    actor_head_kernel_size: 3
    memory_kwargs:
      kind: gru
      hidden_dim: 256
      entity_edge_radius: 2.0
    region_tokenizer_kwargs:
      kind: heuristic
    hierarchical_action_kwargs:
      strategy_latent_dim: 8
      num_groups: 8
  algo_hyperparams:
    <<: *microrts-squnet-map16-algo-defaults
    batch_size: 1536
    n_epochs: 2
    learning_rate: !!float 5e-5
    clip_range: 0.1
    ent_coef: 0.01
    multi_reward_weights: [0.8, 0.01, 0.19]
    vf_coef: [0.5, 0.1, 0.2]
""".strip() + "\n"

marker = '# ---- Kaggle T4x2 smoke/ablation configs appended by kaggle_microrts_t4x2.ipynb ----'
if marker in text:
    text = text[:text.index(marker)].rstrip()
    print('Replacing existing Kaggle configs in', hp_path)
else:
    text = text.rstrip()
    print('Appending Kaggle configs to', hp_path)
hp_path.write_text(text + "\n\n" + KAGGLE_CONFIG, encoding='utf-8')

print('Configured timesteps:', KAGGLE_TIMESTEPS)
print('Primary experiment:', PRIMARY_EXPERIMENT)


In [ ]:
# 4. Fast architecture profiling before spending hours training
from pathlib import Path
KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()
!cd "$WORKDIR" && PYTHONPATH="$WORKDIR:${PYTHONPATH:-}" $KAGGLE_PY scripts/profile_microrts_policy.py   --styles squeeze_unet,hybrid_entity_grid,hierarchical_hybrid_entity_grid   --map-size 16 --batch-size 4 --entities 24   --warmup 5 --iters 20   --action-mode sample


In [ ]:
# 5. Single-GPU primary training. Defaults to the strongest research candidate.
# The subprocess is unbuffered so Kaggle shows rollout/update logs while it runs.
from pathlib import Path
import os, shlex, subprocess

KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['PYTHONUNBUFFERED'] = '1'
env['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
env['PYTHONPATH'] = str(WORKDIR) + ':' + env.get('PYTHONPATH', '')
label = PRIMARY_EXPERIMENT.replace('Microrts-kaggle-', '').replace('-map16-bots', '')

cmd = [
    str(KAGGLE_PY), '-u', 'train.py',
    '--algo', 'ppo',
    '--env', PRIMARY_EXPERIMENT,
    '--seed', '1',
    '--device-indexes', '0',
]
if USE_WANDB:
    cmd += [
        '--wandb-project-name', WANDB_PROJECT,
        '--wandb-tags', 'kaggle', 't4x2', label, 'map16', 'bots', 'primary',
    ]

print('Primary experiment:', PRIMARY_EXPERIMENT, flush=True)
print('Running:', ' '.join(shlex.quote(str(part)) for part in cmd), flush=True)
subprocess.run(cmd, cwd=WORKDIR, env=env, check=True)


## Dual T4 parallel experiments

microRTS is often CPU/Java-env limited, so a single PPO run may not scale linearly across two T4s. The steadier pattern is one independent run per GPU.

Default layout:

- GPU0: `Microrts-kaggle-squnet-map16-bots`, seed 1, proven baseline
- GPU1: `Microrts-kaggle-hierarchical-hybrid-memory-map16-bots`, seed 2, strongest candidate

If Kaggle CPU cannot keep up, change `n_envs` and `num_bot_envs` from 12 to 6 and bots to `coacAI: 3`, `mayari: 3`. If the memory run is unstable, switch GPU1 to `Microrts-kaggle-hybrid-map16-bots` for the lighter ablation.


In [ ]:
# 6. Optional dual-GPU parallel run: one experiment per T4.
# Set RUN_DUAL = True when you are ready.
RUN_DUAL = False

from pathlib import Path
import os, shlex, subprocess, threading

KAGGLE_PY = Path('/kaggle/working/rai_microrts_python.txt').read_text(encoding='utf-8').strip()

def _wandb_args(label):
    if not USE_WANDB:
        return []
    return [
        '--wandb-project-name', WANDB_PROJECT,
        '--wandb-tags', 'kaggle', 't4x2', label, 'map16', 'bots',
    ]

def _stream_log(label, proc, log_path):
    assert proc.stdout is not None
    with open(log_path, 'w', encoding='utf-8', buffering=1) as f:
        for line in proc.stdout:
            f.write(line)
            print(f'[{label}] {line}', end='', flush=True)

if RUN_DUAL:
    log_dir = WORKDIR / 'kaggle_logs'
    log_dir.mkdir(exist_ok=True)
    jobs = [
        ('0', BASELINE_EXPERIMENT, '1', 'squnet'),
        ('1', STRONGEST_EXPERIMENT, '2', 'hierarchical'),
    ]
    procs = []
    for gpu, env_name, seed, label in jobs:
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = gpu
        env['WANDB_MODE'] = 'online' if USE_WANDB else 'disabled'
        env['PYTHONUNBUFFERED'] = '1'
        env['PYTHONPATH'] = str(WORKDIR) + ':' + env.get('PYTHONPATH', '')
        log_path = log_dir / f'{label}-gpu{gpu}-seed{seed}.log'
        cmd = [
            str(KAGGLE_PY), '-u', 'train.py',
            '--algo', 'ppo',
            '--env', env_name,
            '--seed', seed,
            '--device-indexes', '0',
        ] + _wandb_args(label)
        print('Starting', label, 'on physical GPU', gpu, 'log:', log_path, flush=True)
        print('Command:', ' '.join(shlex.quote(str(part)) for part in cmd), flush=True)
        proc = subprocess.Popen(
            cmd,
            cwd=WORKDIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        thread = threading.Thread(target=_stream_log, args=(label, proc, log_path), daemon=True)
        thread.start()
        procs.append((label, proc, thread))

    failures = []
    for label, proc, thread in procs:
        code = proc.wait()
        thread.join()
        print(label, 'exit code:', code, flush=True)
        if code:
            failures.append((label, code))
    if failures:
        raise RuntimeError(f'Dual run failed: {failures}')
else:
    print('RUN_DUAL is False; skipped.')


In [ ]:
# 7. TensorBoard
# If the Kaggle notebook extension fails, the runs/ folder is still saved as an output artifact.
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/rai-microrts-arena/runs


In [ ]:
# 8. Inspect and package outputs
import os
os.chdir(WORKDIR)
!find saved_models -maxdepth 3 -type f | head -50 || true
!tar -czf /kaggle/working/rai_microrts_outputs.tgz saved_models runs videos kaggle_logs 2>/dev/null || true
print('Packed outputs to /kaggle/working/rai_microrts_outputs.tgz')


## Common debugging moves

- `approx_kl` too high: lower `learning_rate` to `5e-5`, or lower `n_epochs` from 4 to 2.
- `entropy` reaches 0 too fast: exploration collapsed; raise `ent_coef` or extend bot warmup.
- `explained_variance` stays low: critic is not learning; check reward scale, `vf_coef`, and `normalize_value_targets`.
- `steps_per_second` is low while GPU is idle: bottleneck is CPU/Java env; tune `n_envs` first, then reduce bot/eval load.
- `hierarchical_hybrid_entity_grid` is much slower than SquNet: check profiler p95, then lower `batch_size` to 1024, set `encoder_layers` to 1, or temporarily fall back to `hybrid_entity_grid`.
- Memory run win rate is noisy: check `approx_kl`, eval/bot mix, and episode reset behavior; use `Microrts-kaggle-hybrid-map16-bots` as the no-memory control.
- SquNet, Hybrid, and Hierarchical all fail to learn: try ACBC warm-start, league/PFSP, or GraphDINO before adding a heavier Mamba branch.
